In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
bn_df = pd.read_csv('bn.csv')
en_df = pd.read_csv('en.csv')

In [ ]:
bn_df.head()

In [ ]:
en_df.head()

In [ ]:
bn_df = bn_df.drop(columns=['category'])
bn_df['label'] = bn_df['label'].astype(int)

bn_df.head()

In [ ]:
en_df = en_df.drop(en_df.columns[0], axis=1)
en_df = en_df.rename(columns={'title': 'headline', 'text': 'content'})
en_df['label'] = en_df['label'].astype(int)

en_df.head()

In [ ]:
bn_df = bn_df.dropna()
bn_df = bn_df.drop_duplicates()
bn_df.shape

In [ ]:

en_df = en_df.dropna()
en_df = en_df.drop_duplicates()
en_df.shape

In [ ]:
en_df['headline'] = en_df['headline'].str.lower()
en_df['content'] = en_df['content'].str.lower()
en_df.head()

In [ ]:
def clean_text(text):
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
bn_df['headline'] = bn_df['headline'].apply(clean_text)
bn_df['content'] = bn_df['content'].apply(clean_text)

bn_df.head()

In [ ]:
en_df['headline'] = en_df['headline'].apply(clean_text)
en_df['content'] = en_df['content'].apply(clean_text)

en_df.head()

In [ ]:
BN_X = bn_df[['headline', 'content']]
BN_Y = bn_df['label']

EN_X = en_df[['headline', 'content']]
EN_Y = en_df['label']

In [ ]:
BN_X_TRAIN, BN_X_TEST, BN_Y_TRAIN, BN_Y_TEST = train_test_split(BN_X, BN_Y, test_size=0.2, random_state=42, stratify=BN_Y)
EN_X_TRAIN, EN_X_TEST, EN_Y_TRAIN, EN_Y_TEST = train_test_split(EN_X, EN_Y, test_size=0.2, random_state=42, stratify=EN_Y)

In [ ]:
BN_X_TRAIN['text'] = BN_X_TRAIN['headline'] + ' ' + BN_X_TRAIN['content']
BN_X_TEST['text'] = BN_X_TEST['headline'] + ' ' + BN_X_TEST['content']

EN_X_TRAIN['text'] = EN_X_TRAIN['headline'] + ' ' + EN_X_TRAIN['content']
EN_X_TEST['text'] = EN_X_TEST['headline'] + ' ' + EN_X_TEST['content']

In [ ]:
en_vectorizer = TfidfVectorizer(max_features=10000)
bn_vectorizer = TfidfVectorizer(max_features=10000)

bn_vectorizer.fit(BN_X_TRAIN['text'])
en_vectorizer.fit(EN_X_TRAIN['text'])

BN_X_TRAIN_TFIDF = bn_vectorizer.transform(BN_X_TRAIN['text'])
BN_X_TEST_TFIDF = bn_vectorizer.transform(BN_X_TEST['text'])

EN_X_TRAIN_TFIDF = en_vectorizer.transform(EN_X_TRAIN['text'])
EN_X_TEST_TFIDF = en_vectorizer.transform(EN_X_TEST['text'])

In [ ]:
bn_model = LogisticRegression(max_iter=1000)
bn_model.fit(BN_X_TRAIN_TFIDF, BN_Y_TRAIN)

en_model = LogisticRegression(max_iter=1000)
en_model.fit(EN_X_TRAIN_TFIDF, EN_Y_TRAIN)

In [ ]:
import pickle

with open('bn_model.pkl', 'wb') as f:
    pickle.dump(bn_model, f)

with open('en_model.pkl', 'wb') as f:
    pickle.dump(en_model, f)

with open('bn_vectorizer.pkl', 'wb') as f:
    pickle.dump(bn_vectorizer, f)

with open('en_vectorizer.pkl', 'wb') as f:
    pickle.dump(en_vectorizer, f)

In [ ]:
BN_Y_PRED = bn_model.predict(BN_X_TEST_TFIDF)
accuracy_score(BN_Y_TEST, BN_Y_PRED)

In [ ]:
EN_Y_PRED = en_model.predict(EN_X_TEST_TFIDF)
accuracy_score(EN_Y_TEST, EN_Y_PRED)